# Day 4: Support Vector Machines (SVMs)

## Overview

SVMs find maximum-margin hyperplanes to maximize generalization. With non-linear kernels, SVMs can fit complex patterns. Learn when to use SVMs vs tree-based methods and how to tune them effectively.

## Learning Objectives

- Understand maximum margin principle
- Learn soft margins and regularisation (C parameter)
- Experiment with linear, RBF, and polynomial kernels
- Understand when SVMs outperform tree methods

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Linear SVM: Maximum Margin Principle

In [ ]:
# Create a simple 2D linearly separable dataset
np.random.seed(42)
X = np.vstack([
    np.random.normal(loc=0, scale=0.5, size=(50, 2)),
    np.random.normal(loc=3, scale=0.5, size=(50, 2))
])
y = np.hstack([np.zeros(50), np.ones(50)])

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train linear SVM
svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X_scaled, y)

# Plot decision boundary
fig, ax = plt.subplots(figsize=(10, 8))

# Plot points
ax.scatter(X_scaled[y == 0, 0], X_scaled[y == 0, 1], c='blue', s=50, label='Class 0')
ax.scatter(X_scaled[y == 1, 0], X_scaled[y == 1, 1], c='red', s=50, label='Class 1')

# Plot decision boundary
h = 0.02
xx, yy = np.meshgrid(np.arange(X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1, h),
                       np.arange(X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1, h))
Z = svm_linear.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
ax.scatter(svm_linear.support_vectors_[:, 0], svm_linear.support_vectors_[:, 1],
           s=200, linewidth=1.5, facecolors='none', edgecolors='green', label='Support Vectors')

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Linear SVM: Maximum Margin Hyperplane')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Accuracy: {svm_linear.score(X_scaled, y):.4f}")
print(f"Number of support vectors: {len(svm_linear.support_vectors_)}")

## 2. Soft Margin and C Parameter

In [ ]:
# Load Iris and prepare
iris = load_iris()
X_iris = iris.data
y_iris = (iris.target == 2).astype(int)  # Binary classification

X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# Scale
scaler_iris = StandardScaler()
X_train_scaled = scaler_iris.fit_transform(X_train)
X_test_scaled = scaler_iris.transform(X_test)

# Test different C values
C_values = [0.01, 0.1, 1.0, 10, 100]
train_scores = []
test_scores = []
n_support_vectors = []

for C in C_values:
    svm = SVC(kernel='linear', C=C)
    svm.fit(X_train_scaled, y_train)
    train_scores.append(svm.score(X_train_scaled, y_train))
    test_scores.append(svm.score(X_test_scaled, y_test))
    n_support_vectors.append(len(svm.support_vectors_))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(C_values, train_scores, marker='o', label='Train', linewidth=2)
ax1.plot(C_values, test_scores, marker='s', label='Test', linewidth=2)
ax1.set_xlabel('C (Regularisation Parameter)')
ax1.set_ylabel('Accuracy')
ax1.set_xscale('log')
ax1.set_title('Effect of C on Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(C_values, n_support_vectors, marker='o', color='orange', linewidth=2)
ax2.set_xlabel('C (Regularisation Parameter)')
ax2.set_ylabel('Number of Support Vectors')
ax2.set_xscale('log')
ax2.set_title('Effect of C on Support Vectors')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Low C (high regularisation): wider margin, more support vectors")
print(f"High C (low regularisation): narrow margin, fewer support vectors")

## 3. Kernels: Linear vs RBF vs Polynomial

In [ ]:
# Test different kernels
kernels = ['linear', 'rbf', 'poly']
kernel_scores = {}

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, gamma='scale', degree=3)
    svm.fit(X_train_scaled, y_train)
    train_score = svm.score(X_train_scaled, y_train)
    test_score = svm.score(X_test_scaled, y_test)
    kernel_scores[kernel] = {'train': train_score, 'test': test_score}
    print(f"Kernel: {kernel:10s} | Train: {train_score:.4f} | Test: {test_score:.4f}")

# Visualize
kernels_list = list(kernel_scores.keys())
train_vals = [kernel_scores[k]['train'] for k in kernels_list]
test_vals = [kernel_scores[k]['test'] for k in kernels_list]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(kernels_list))
width = 0.35
ax.bar(x - width/2, train_vals, width, label='Train', alpha=0.8)
ax.bar(x + width/2, test_vals, width, label='Test', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(kernels_list)
ax.set_ylabel('Accuracy')
ax.set_title('SVM Kernel Comparison')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4. RBF Kernel: Tuning Gamma

In [ ]:
# Test different gamma values for RBF kernel
gamma_values = [0.001, 0.01, 0.1, 1.0, 'scale', 'auto']
rbf_scores = []

for gamma in gamma_values:
    svm_rbf = SVC(kernel='rbf', C=1.0, gamma=gamma)
    svm_rbf.fit(X_train_scaled, y_train)
    test_score = svm_rbf.score(X_test_scaled, y_test)
    rbf_scores.append(test_score)
    print(f"Gamma: {str(gamma):10s} | Test Accuracy: {test_score:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(gamma_values)), rbf_scores)
ax.set_xticks(range(len(gamma_values)))
ax.set_xticklabels([str(g) for g in gamma_values], rotation=45)
ax.set_ylabel('Test Accuracy')
ax.set_title('RBF Kernel: Effect of Gamma')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nGamma controls RBF kernel width: large gamma = complex model, small gamma = simple model")

## 5. SVM vs Other Algorithms

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import time

# Create larger dataset for timing comparison
X_large, y_large = make_classification(n_samples=5000, n_features=20, random_state=42)
X_train_lg, X_test_lg, y_train_lg, y_test_lg = train_test_split(
    X_large, y_large, test_size=0.3, random_state=42
)

# Scale for SVM
scaler_lg = StandardScaler()
X_train_lg_scaled = scaler_lg.fit_transform(X_train_lg)
X_test_lg_scaled = scaler_lg.transform(X_test_lg)

algorithms = [
    ('SVM (RBF)', SVC(kernel='rbf', C=1.0), X_train_lg_scaled, X_test_lg_scaled),
    ('Random Forest', RandomForestClassifier(n_estimators=100), X_train_lg, X_test_lg),
    ('KNN', KNeighborsClassifier(n_neighbors=5), X_train_lg_scaled, X_test_lg_scaled)
]

results = []
for name, model, X_tr, X_te in algorithms:
    start = time.time()
    model.fit(X_tr, y_train_lg)
    train_time = time.time() - start
    
    start = time.time()
    score = model.score(X_te, y_test_lg)
    pred_time = time.time() - start
    
    results.append({'Algorithm': name, 'Train Time': train_time, 'Pred Time': pred_time, 'Accuracy': score})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nSVM: Good for interpretability, small-medium data. Trees: Good for speed and scale.")

## Exercises

1. Scale and train an SVM without StandardScaler. Compare accuracy to scaled SVM.
2. Use GridSearchCV to find optimal C and gamma for RBF SVM on Iris.
3. Train linear, RBF, and poly SVMs on a non-linear dataset. Which kernel wins?

## Solutions

Key insights:
- Always scale SVM inputs (distance-based algorithm)
- Low C = wider margin = underfitting; High C = narrow margin = overfitting
- RBF kernel good for non-linear patterns
- SVM slower than trees on large datasets